# How I'd approach this in 2026 — CLIP & modern embeddings

> **Context.** The original project (2023) used Word2Vec, BERT and VGG16 — solid
> for its time. The field moved fast. This notebook is a **design note**, not a
> re-run: what I would change today and *why*, with illustrative code. It is the
> "senior perspective" companion to notebooks 01–02.

---

### TL;DR — three upgrades

| 2023 (original) | 2026 (what I'd do) | Why it's better |
|---|---|---|
| Word2Vec / BERT `[CLS]`, frozen | **Sentence-Transformers** (e.g. `all-MiniLM`, `bge`) | Embeddings *designed* for sentence similarity, not repurposed |
| VGG16 penultimate layer | **DINOv2 / modern ViT** or a CLIP image tower | Stronger, more transferable visual features |
| Two separate encoders + late fusion | **CLIP — one shared text+image space** | Native multimodal: text and image embeddings are directly comparable |


## 1 · The core idea: a shared embedding space

In 2023 the hard part was *fusing* two unrelated feature spaces (Word2Vec vs
VGG16) — hence the early-vs-late fusion study in notebook 02. **CLIP removes the
problem by construction:** it is trained so that an image and its text land at
the *same place* in one vector space. Classification, retrieval and even
zero-shot labelling then become cosine-similarity lookups.

In [ ]:
# Illustrative — requires: pip install sentence-transformers torch
# from sentence_transformers import SentenceTransformer
# clip = SentenceTransformer("clip-ViT-B-32")
#
# txt_emb = clip.encode(df["description"].tolist())          # (n, 512)
# img_emb = clip.encode([Image.open(p) for p in image_paths]) # (n, 512)  — SAME space!
#
# A product's text and its photo are now directly comparable (cosine).

## 2 · Zero-shot classification — *no training at all*

With CLIP you can classify into the 7 categories **without a single labelled
example**: embed the category names as text, embed each product (text and/or
image), and assign the nearest category. In 2023 this needed K-Means + manual
alignment; in 2026 it is three lines.

In [ ]:
# label_prompts = [f"a product in the category: {c}" for c in cats]
# label_emb = clip.encode(label_prompts)                      # (7, 512)
#
# sims = img_emb @ label_emb.T                                # cosine similarities
# pred = sims.argmax(axis=1)                                  # zero-shot prediction
# -> expected to rival the 2023 *supervised* pipeline, with zero labels

## 3 · True multimodal fusion — trivial in a shared space

Because text and image embeddings live in one space, "fusion" is just an
average (or a concatenation that finally makes sense), then a light classifier —
or again zero-shot. No more early-vs-late dilemma.

In [ ]:
# fused = (txt_emb + img_emb) / 2        # meaningful: same space, same scale
# LogisticRegression().fit(fused[train], y[train])   # a strong, tiny head

## 4 · What I would keep from 2023

Not everything should change — the *method* still holds:
- the **fair, shared evaluation protocol** across approaches;
- **fixed seeds** and a held-out test set for reproducibility;
- **late fusion at the decision level** remains a strong, cheap baseline;
- reporting the **negative result** (naïve concat) honestly.

Modern models change the *tools*, not the *rigour*.

## 5 · Expected payoff & honest caveats

- **Likely higher accuracy** and, more importantly, **zero-shot** capability —
  add a new category without retraining, just by writing its name.
- **One model instead of three** (CLIP replaces Word2Vec + BERT + VGG16).
- **Caveat:** CLIP is English-centric and web-biased; on niche e-commerce
  categories a short fine-tune (LoRA) or a domain-specific model may still win.
  The 2023 pipeline remains a valid, dependency-light baseline — which is exactly
  why it is worth keeping as the comparison point.
